# Naturalistic EMA prediction based on ACC - Dyskinesia use case

## 0) Imports

- document versions for reproducibility

In [ ]:
# import packages
import datetime as dt
import pandas as pd
import numpy as np
import os
import sys
import csv
import json
import importlib
from itertools import product, compress
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, variation

from dataclasses import dataclass, field


In [ ]:
print('Python sys', sys.version)
print('pandas', pd.__version__)
print('numpy', np.__version__)
# print('mne_bids', mne_bids.__version__)
# print('mne', mne.__version__)
# print('sci-py', scipy.__version__)
# print('sci-kit learn', sk.__version__)
# print('matplotlib', plt_version)

"""
Python sys 3.11.5 | packaged by Anaconda, Inc. | (main, Sep 11 2023, 13:26:23) [MSC v.1916 64 bit (AMD64)]
pandas 2.1.1
numpy 1.26.0

from 16.09

Python sys 3.11.5 | packaged by Anaconda, Inc. | (main, Sep 11 2023, 13:26:23) [MSC v.1916 64 bit (AMD64)]
pandas 2.3.2
numpy 2.3.3
"""

Import custom functions

In [ ]:
# from dbs_home repo
from dbs_home.load_raw.main_load_raw import loadSubject 
import dbs_home.utils.helpers as home_helpers
import dbs_home.utils.ema_utils as home_ema_utils
import dbs_home.plot_data.plot_compliance as plot_home_compl
import dbs_home.preprocessing.preparing_ema as home_ema_prep

from dbs_home.preprocessing import get_submovements
import dbs_home.preprocessing.acc_preprocessing as acc_prep
import dbs_home.preprocessing.submovement_processing as submove_proc
import dbs_home.load_raw.load_watch_raw as load_watch


In [ ]:
# from current repo
from utils import load_utils, load_data, prep_data
import utils.acc_features as acc_fts
import utils.ft_extract_class as ft_extr_class
import utils.feat_extraction as ft_extr
import utils.data_handling_ema_acc as data_handling

from plotting import plot_help



## 1) Load and check naturalistic data

In [ ]:
# set paths

feas_data_path = os.path.join(
    os.path.dirname(load_utils.get_onedrive_path()),
    'PROJECTS', 'home_feasibility'
)
feas_fig_path = os.path.join(
    load_utils.get_onedrive_path('figures'),
    'feasibility'
)



#### Load ACC data
- create SVM
- filter data within the dataclass

In [ ]:
# LID
sub_id = 'hm24'
ses_id = 'ses03'

FT_PARAMS_VERSION = 'v3'  # v3 updated lid version

In [ ]:
# import naturalistic data via dbs_home repo




### test days for hm24-ses01  # dyskinesia
# dev_day_selection = ['2025-07-17', '2025-07-18']
# dev_day_selection = [f'2025-07-{d}' for d in np.arange(17, 31)]
dev_day_selection = []

### test days for hm20-ses01  # tremor
# dev_day_selection = [
#     '2025-06-13', '2025-06-14',
#     '2025-06-15', '2025-06-16'
# ]

home_dat = loadSubject(
    sub=sub_id,
    ses=ses_id,
    incl_STEPS=False,
    incl_EPHYS=False,
    incl_EMA=True,
    incl_ACC=True,
    day_selection=dev_day_selection
)

Check available EMAs

In [ ]:
plot_home_compl.plot_EMA_completion_perSession(home_dat)

## 2) ACC processing incl. Feature extraction

#### 2a) Extract features from Acc-Windows aligned to EMAs

In [ ]:
importlib.reload(ft_extr_class)
importlib.reload(acc_fts)
importlib.reload(data_handling)
importlib.reload(ft_extr)


xy_dict = {}

# iter_settings = {
#     'nosm_allwindow':[False, True, True],
#     'sm_merged': [True, False, False],
#     'sm_single': [True, False, True]}

ft_settings = {
    'nosm_allwindow': {
        'EXTRACT_FT_FROM_SMs': False,
        'EXTRACT_FT_FULL_WIN': True,
        'ACC_FEATS_on_SINGLE_MOVES': True
    },
    'sm_merged': {
        'EXTRACT_FT_FROM_SMs': True,
        'EXTRACT_FT_FULL_WIN': False,
        'ACC_FEATS_on_SINGLE_MOVES': False
    },
    'sm_single': {
        'EXTRACT_FT_FROM_SMs': True,
        'EXTRACT_FT_FULL_WIN': False,
        'ACC_FEATS_on_SINGLE_MOVES': True
    }
}




for ft_type, ft_settings in ft_settings.items():

    xy_dict[ft_type] = ft_extr.get_features_per_session(
        home_dat=home_dat,
        sub_id=sub_id,
        ses_id=ses_id,
        FT_PARAMS_VERSION=FT_PARAMS_VERSION,
        LOAD_SAVE_FEATS = True,
        # define how features should be extracted
        STORE_SUBMOVES = False,
        # plotting settings
        SAVE_PLOT = False,
        SHOW_PLOT = False,
        **ft_settings
    )

In [ ]:
[f'{k}: {v.shape}, column-names: {list(v.keys())}' for k, v in xy_dict.items()]

#### 2b) feature extractions for full days

updated function to directly get feature dataframes, without home_dat requirement

In [ ]:
importlib.reload(acc_prep)
importlib.reload(load_watch)
importlib.reload(ft_extr)


In [ ]:
importlib.reload(ft_extr)

dfs_sessions = {}

sub_id = 'hm24'

for ses_id in ['ses01', 'ses02']:
    
    tempdf = ft_extr.get_feat_df_for_pred(
        sub_id=sub_id, ses_id=ses_id,
        ft_set_sel = 'sm_single',
        ONLY_EMA_WINDOWS=False,
    )
    dfs_sessions[ses_id] = tempdf


## 3a) Evaluate extracted Features

- Hssayeni et al, Scientific Reports 2021
    - strongest wrist-features: angular velocity, standard deviation, power of secondary frequency, power of 1–4 Hz band, and Shannon Entropy (r = 0.82  - r = 0.75)

- from svm: classic features
- include cross-corr between pc1 and pc2


In [ ]:
from plotting import plot_home_preds as plt_preds
import plotting.plot_ft_explor as plt_fts

define selected data

In [ ]:
SES = 'ses01'
FT_TYPE = 'sm_merged'    # nosm_allwindow, sm_merged, sm_single
FT_PARAMS_VERSION = 'v3'  # v3 updated lid version
EMA_ref = 'LID'



In [ ]:
importlib.reload(ft_extr)


xy_dict = {}

for ses_id in ['ses01', 'ses02', 'ses03']:
    
    tempdf = ft_extr.get_feat_df_for_pred(
        sub_id=sub_id,
        ses_id=ses_id,
        ft_set_sel=FT_TYPE,
        FT_PARAMS_VERSION=FT_PARAMS_VERSION,
        ONLY_EMA_WINDOWS=True,
        # **ft_settings[FT_TYPE]
    )
    xy_dict[ses_id] = tempdf

In [ ]:
ft_df = xy_dict[SES].copy()


check selected variables in dataframe and first rows of dataframe

In [ ]:
print(ft_df.shape)
print(ft_df.keys())
print(ft_df.head())
print([f'LID "{k}": {v} counts' for k, v in zip(
    *np.unique(ft_df[EMA_ref], return_counts=True))])


In [ ]:
importlib.reload(ft_extr)

# define which features to include in prediction and which to exclude (times, ema)
feats_incl, EMA_CODING = ft_extr.get_feat_params(FT_PARAMS_VERSION)
keys_excl = ['timestamp'] + list(EMA_CODING.keys())  # times + ema

Visualize feature distributions

In [ ]:
importlib.reload(plt_fts)

save_plot = True


for ft_name in ft_df.keys():
    print(f'\n{ft_name}')
    # if ft_name not in ft_df.keys():
    #     print(f'Feature {ft_name} not found in dataframe columns. Skipping.')
    #     continue
    if ft_name in keys_excl:
        print(f'Feature {ft_name} is not included in the feature set. Skipping.')
        continue
    
    ### plot distributions with histo, scatter, and boxes
    plt_fts.plot_ft_distribution(
        ft_df=ft_df,
        ft_name=ft_name,
        EMA_ref=EMA_ref,
        sub_id=sub_id,
        SES=SES,
        FT_TYPE=FT_TYPE,
        FT_PARAMS_VERSION=FT_PARAMS_VERSION,
        save_plot=save_plot,
    )



## 3b) Predict EMAs based on Wearable-features

#### LID-classification: cross-validation based on preop session

- only EMA windows
- cv-folds: n-folds of EMA-windows from all sessions

In [ ]:

from sklearn.model_selection import GridSearchCV, StratifiedKFold, KFold
from sklearn.metrics import r2_score, root_mean_squared_error

from sklearn.linear_model import ElasticNet, Lasso, Ridge, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from utils import pred_nat_ema
import utils.pred_utils as pred_utils

In [ ]:
# define window for cross-validation
cv_df = xy_dict[SES].copy()
print(cv_df.keys())

EMA_Y = 'LID'

EXCL_HR = False  # exclude heart rate features (not available in all timepoints)
TRANSFORM_Y = False  # transform LID into 3 classes: 1, 2 (1-4), 3 (>4)

In [ ]:
### select features to include in prediction
if EXCL_HR:
    PRED_FTS = [
        k for k in list(cv_df.keys())
        if (k not in keys_excl) and ('hr' not in k)
    ]
else:
    PRED_FTS = [
        k for k in list(cv_df.keys()) if k not in keys_excl
    ]
print(f'model trained on features: {PRED_FTS}')

In [ ]:
importlib.reload(pred_utils)

### define training X, y
X_all = cv_df[PRED_FTS].values.copy()
y_all = cv_df[EMA_Y].values.copy().astype(float)

### prepare data
skew_corr_bool_list = [pred_utils.check_skewness(
    X_all[:, i][~np.isnan(X_all[:, i])],
    threshold=1.5,
)[0] for i in range(X_all.shape[1])]
X_all, y_all = pred_utils.prepare_Xy_for_regression(
    X_all, y_all,
    remove_nan_rows=True,
    transform_skewed_feats=True,
    verbose=True,
)

In [ ]:
# pipe includes scaling based on training data, and regression model
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('reg', Ridge(alpha=1.0)),
    # ('reg', LogisticRegression(multi_class='multinomial',)),
])


In [ ]:
y_pred_all = np.zeros_like(y_all)

n_folds = 5
cv_folds = KFold(n_splits=n_folds).split(X_all)

for i_fold, (train_idx, test_idx) in enumerate(cv_folds):
    print(f'test indices for fold {i_fold + 1}: {test_idx}')
    X_train, X_test = X_all[train_idx], X_all[test_idx]
    y_train, y_test = y_all[train_idx], y_all[test_idx]

    # train regression model
    pipe.fit(X_train, y_train)
    # predict test set (incl scaling of fts based on train set)
    y_pred = pipe.predict(X_test)
    # clip predictions to valid range and round to nearest integer
    y_pred = np.clip(np.round(y_pred), y_train.min(), y_train.max()).astype(int)
    y_pred_all[test_idx] = y_pred  # add predictions to total array in correct indices
    
    print(f'finished fold {i_fold + 1} / {n_folds}')

In [ ]:
from scipy.stats import spearmanr, kendalltau
from sklearn.metrics import mean_absolute_error, cohen_kappa_score


quick evaluation, check sign with permutations

In [ ]:
rho, p = spearmanr(y_all, y_pred_all)
print(f"Spearman rho={rho.round(2)}, p={p.round(5)}")

tau, p_tau = kendalltau(y_all, y_pred_all)
print(f"Kendall tau={tau.round(2)}, p={p_tau.round(5)}")  

mae = mean_absolute_error(y_all, y_pred_all)
kappa = cohen_kappa_score(y_all, y_pred_all, weights='linear')
print(f"Mean Absolute Error: {mae:.2f}, Cohen's Kappa: {kappa:.2f}")

r2 = r2_score(y_all, y_pred_all)
rmse = root_mean_squared_error(y_all, y_pred_all)
print(f"### R2 for all data: {r2:.2f}, RMSE: {rmse:.2f}")

cv_results = {
    'spearman_rho': rho,
    'spearman_p': p,
    'kendall_tau': tau,
    'kendall_p': p_tau,
    'mae': mae,
    'kappa': kappa,
    'r2': r2,
    'rmse': rmse
}


fig, ax = plt.subplots(1, 1, figsize=(6, 6))

y_jitter = np.random.normal(0, 0.1, size=y_all.shape)  # add jitter for better visibility
ax.scatter(y_all + y_jitter, y_pred_all, alpha=0.5)
ax.set_xlabel('True LID')
ax.set_ylabel('Predicted LID')

plt.show()

## 3c) Check LID classifier on post-op sessions

- Train-split: EMA-windows ses01, Test-split: EMA-windows ses02 and ses03

In [ ]:
cv_df = xy_dict['ses01'].copy()
test_df = pd.concat([xy_dict['ses02'], xy_dict['ses03']]).reset_index(drop=True).copy()
# test_df = xy_dict['ses03'].copy()

In [ ]:
### prepare train and test data
X_train = cv_df[PRED_FTS].values.copy()
y_train = cv_df[EMA_Y].values.copy().astype(float)
X_test = test_df[PRED_FTS].values.copy()
y_test = test_df[EMA_Y].values.copy().astype(float)

### prepare data

# check which features will be log-transformed in training
skew_corr_bool_list = [pred_utils.check_skewness(
    X_all[:, i][~np.isnan(X_all[:, i])],
    threshold=1.5,
)[0] for i in range(X_all.shape[1])]

print('training data:')
X_train, y_train = pred_utils.prepare_Xy_for_regression(
    X_train, y_train,
    remove_nan_rows=True,
    transform_skewed_feats=True,
    verbose=True,
)

print('test data:')
X_test, y_test = pred_utils.prepare_Xy_for_regression(
    X_test, y_test,
    remove_nan_rows=True,
    transform_skewed_feats=False,
    verbose=True,
)
# only log-transform test-features that were log-transformed in training
for i in range(X_test.shape[1]):
    if skew_corr_bool_list[i]:  # if this feature was log-transformed in training
        X_test[:, i] = pred_utils.log_transf(X_test[:, i])

In [ ]:
# pipe includes scaling based on training data, and regression model
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('reg', Ridge(alpha=1.0)),
    # ('reg', LogisticRegression(multi_class='multinomial',)),
])

validation test of true predictions

In [ ]:
# train regression model
pipe.fit(X_train, y_train)
# predict test set (incl scaling of fts based on train set)
y_pred = pipe.predict(X_test)
# clip predictions to valid range and round to nearest integer
y_pred = np.clip(np.round(y_pred), y_train.min(), y_train.max()).astype(int)

In [ ]:
rho, p = spearmanr(y_test, y_pred)
print(f"Spearman rho={rho:.2f}, p={p:.5f}")

tau, p_tau = kendalltau(y_test, y_pred)
print(f"Kendall tau={tau:.2f}, p={p_tau:.5f}")  

mae = mean_absolute_error(y_test, y_pred)
kappa = cohen_kappa_score(y_test, y_pred, weights='linear')
print(f"Mean Absolute Error: {mae:.2f}, Cohen's Kappa: {kappa:.2f}")

r2 = r2_score(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred)
print(f"### R2 for all data: {r2:.2f}, RMSE: {rmse:.2f}")


test_results = {
    'spearman_rho': rho,
    'spearman_p': p,
    'kendall_tau': tau,
    'kendall_p': p_tau,
    'mae': mae,
    'kappa': kappa,
    'r2': r2,
    'rmse': rmse
}


fig, ax = plt.subplots(1, 1, figsize=(6, 6))

# y_jitter = np.random.normal(0, 0.1, size=y_test.shape)  # add jitter for better visibility
# ax.scatter(y_test + y_jitter, y_pred + y_jitter, alpha=0.5)

# boxplots of predicted value per true LID class
data_to_plot = [y_pred[y_test == lid] for lid in np.unique(y_test)]
ax.boxplot(data_to_plot, labels=np.unique(y_test))

ax.set_xlabel('True LID')
ax.set_ylabel('Predicted LID')

plt.show()

Test statistical significance with permutations

In [ ]:
n_perm = 1000
np.random.seed(27)  # for reproducibility

perm_results = {
    'rho': [],
    'tau': [],
    'mae': [],
    'kappa': [],
    'r2': [],
    'rmse': []
}

for i in range(n_perm):
    print(f'Permutation {i + 1} / {n_perm}')
    y_perm_train = y_train.copy()
    np.random.shuffle(y_perm_train)

    # take random n-samples of values 1 to 9, with free duplication
    # y_perm_train = np.random.choice(
    #     np.arange(1, 10), size=y_train.shape[0], replace=True,
    # )

    # train regression model with permuted training labels
    pipe.fit(X_train, y_perm_train)
    # predict test set (incl scaling of fts based on train set)
    y_pred_perm = pipe.predict(X_test)
    # clip predictions to valid range and round to nearest integer
    y_pred_perm = np.clip(np.round(y_pred_perm), y_train.min(), y_train.max()).astype(int)

    # calculate metrics for permuted labels
    perm_results['rho'].append(spearmanr(y_test, y_pred_perm)[0])
    perm_results['tau'].append(kendalltau(y_test, y_pred_perm)[0])
    perm_results['mae'].append(mean_absolute_error(y_test, y_pred_perm))
    perm_results['kappa'].append(cohen_kappa_score(y_test, y_pred_perm, weights='linear'))
    perm_results['r2'].append(r2_score(y_test, y_pred_perm))
    perm_results['rmse'].append(root_mean_squared_error(y_test, y_pred_perm))
    

In [ ]:
# plot histograms of permuted metrics with observed metric as vertical line

incl_metrics = ['mae', 'rmse']

fig, axes = plt.subplots(1, len(incl_metrics), figsize=(5 * len(incl_metrics), 5))

for i, metric in enumerate(incl_metrics):

    # calculate p-value as proportion of permuted metrics that are as extreme as or more extreme than the observed metric
    if metric in ['rho', 'tau', 'kappa', 'r2']:  # for metrics where higher is better, p-value is proportion of permuted metrics >= observed metric
        p_value = np.mean(np.array(perm_results[metric]) >= test_results[metric])
    else:  # for metrics where lower is better, p-value is proportion of permuted metrics <= observed metric
        p_value = np.mean(np.array(perm_results[metric]) <= test_results[metric])
    print(f'{metric}: observed={test_results[metric]:.2f}, p-value={p_value:.4f}')

    ax = axes[i]
    ax.hist(perm_results[metric], bins=30,
            color='blue', alpha=.5, label='Permuted',)
    obs_metric = test_results[metric]  # get observed metric value from variable
    ax.axvline(obs_metric, color='red', linestyle='--', label=f'Observed {metric}={obs_metric:.2f}')
    ax.set_title(f'Permutation distribution of {metric}')
    ax.set_xlabel(metric)
    ax.set_ylabel('Frequency')
    ax.legend()
    
plt.tight_layout()
plt.show()

additional visualizations/evaluations for full day windows

In [ ]:
SES_COLORS = {'ses01': 'orange', 'ses02': 'violet', 'ses03': 'darkcyan'}


In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6, 3))

ax.boxplot([v for v in list(test_pred_dict.values())])

ax.set_ylabel('Dyskinesia prediction\n(EMA scale)')
ax.set_xticklabels(list(test_pred_dict.keys()))

plt.show()

for comp_ses in ['ses01', 'ses02']:
    S, p = mannwhitneyu(test_pred_dict[comp_ses], test_pred_dict['ses03'])

    print(f'optimized-DBS vs {comp_ses}: stat = {S.round(2)}, p = {p.round(4)}')

S, p = mannwhitneyu(test_pred_dict['ses01'], test_pred_dict['ses02'])

print(f'pre-op vs not-optimal-DBS: stat = {S.round(2)}, p = {p.round(4)}')

2) Plot predictions versus l-dopa intake moments


In [ ]:
### subplots per ses

FS=14

fig, axes = plt.subplots(len(test_pred_dict), 1,
                         figsize=(3*len(test_pred_dict), 8),)

for i_ses, ses_id in enumerate(test_pred_dict):

   test_stamps = test_stamps_dict[ses_id]
   test_pred = test_pred_dict[ses_id]


   pred_dopa_dist = data_handling.get_intervals_to_ldopa(test_stamps, sub=sub_id, ses=ses_id,)
   (box_distance_groups,
    dist_labels) = data_handling.sort_values_into_ldopa_distances(
      values=test_pred, dopa_distances=pred_dopa_dist,
   )


   bp = axes[i_ses].boxplot(box_distance_groups, patch_artist=True,)
   for patch in bp['boxes']:
      patch.set_facecolor(SES_COLORS[ses_id])
      patch.set_alpha(.4)

   axes[i_ses].set_xlabel('Time vs L-Dopa Intake (minutes)', fontsize=FS,)
   axes[i_ses].set_xticklabels(dist_labels,)
   axes[i_ses].set_ylabel('Dyskinesia prediction\n(EMA scale)', fontsize=FS,)
   axes[i_ses].set_ylim(-2, 9)
   axes[i_ses].set_yticks([1, 3, 5, 7, 9])
   axes[i_ses].set_yticklabels([1, 3, 5, 7, 9])
   axes[i_ses].tick_params(size=FS, labelsize=FS,)

   axes[i_ses].set_title(ses_id, fontsize=FS,)

plt.tight_layout()

figname = 'lidPred_hm24_allSess_ldopaIntakeTime'
# plt.savefig(os.path.join(ntrl_fig_path, 'proof_kin_pred', figname),
#             facecolor='w', dpi=300,)

plt.close()

In [ ]:
### all sessions next to each other

FS=14
pos_adj = [0., .25, .5]

fig, ax = plt.subplots(1, 1, figsize=(9, 3),)

for i_ses, ses_id in enumerate(test_pred_dict):

   test_stamps = test_stamps_dict[ses_id]
   test_pred = test_pred_dict[ses_id]


   pred_dopa_dist = data_handling.get_intervals_to_ldopa(test_stamps, sub=sub_id, ses=ses_id,)
   (box_distance_groups,
    dist_labels) = data_handling.sort_values_into_ldopa_distances(
      values=test_pred, dopa_distances=pred_dopa_dist,
   )

   bp = ax.boxplot(
      box_distance_groups, patch_artist=True, label=ses_id,
      positions=np.arange(len(box_distance_groups)) + pos_adj[i_ses],
      widths=.25,
   )

   for patch in bp['boxes']:
      patch.set_facecolor(SES_COLORS[ses_id])
      patch.set_alpha(.4)

ax.set_xlabel('Time vs L-Dopa Intake (minutes)', fontsize=FS,)
ax.set_xticks(np.arange(len(dist_labels))+pos_adj[1],)
ax.set_xticklabels(dist_labels,)
ax.set_ylabel('Dyskinesia prediction\n(EMA scale)', fontsize=FS,)
ax.set_ylim(-3, 9)
ax.set_yticks([-3, -1, 1, 3, 5, 7, 9])
ax.set_yticklabels([-3, -1, 1, 3, 5, 7, 9])
ax.tick_params(size=FS, labelsize=FS,)

ax.legend(ncol=3, frameon=False, fontsize=FS,
          loc='lower center', bbox_to_anchor=(.5, 1))

plt.tight_layout()

figname = 'lidPred_hm24_allSess_ldopaIntakeTime'
plt.savefig(os.path.join(ntrl_fig_path, 'proof_kin_pred', figname),
            facecolor='w', dpi=300,)

plt.close()

In [ ]:
### daily plot
FS=14

fig, axes = plt.subplots(len(test_pred_dict), 1,
                         figsize=(3*len(test_pred_dict), 8),
                         sharex='col',)

for i_ses, ses_id in enumerate(test_pred_dict.keys()):

   test_stamps = test_stamps_dict[ses_id]
   test_pred = test_pred_dict[ses_id]
   (
        daily_minutes, daily_mean, daily_std
    ) = data_handling.get_ft_daily_mean(test_pred, test_stamps,)
   
   plot_home_preds.plot_daily_ft_mean(
        daily_minutes, daily_mean, daily_std, 'LID prediction',
        use_ax=axes[i_ses], FS=FS, plot_color=SES_COLORS[ses_id]
   )

   # edit subplot
   axes[i_ses].set_title(f'Session: {ses_id}', fontsize=FS,)
   axes[i_ses].set_ylabel('LID prediction\n(EMA scale)', fontsize=FS,)
   axes[i_ses].set_ylim(1, 9)
   axes[i_ses].set_yticks([1, 3, 5, 7, 9])
   axes[i_ses].tick_params(labelsize=FS, size=FS,)


axes[-1].set_xlabel('Time at Day (hours)', fontsize=FS,)

plt.tight_layout()

figname = 'lidPred_hm24_allSess_daytime'
# plt.savefig(os.path.join(ntrl_fig_path, 'proof_kin_pred', figname),
#             facecolor='w', dpi=300,)

plt.close()

# plt.show()

Daytime blocks

In [ ]:
DAY_HOUR_BLOCKS = {'morning': [8, 12],
                   'afternoon1': [12, 15],
                   'afternoon2': [15, 18], 
                   'evening': [18, 22]}

box_lists = {k: {k2: [] for k2 in DAY_HOUR_BLOCKS}
             for k in test_pred_dict}


for ses_id in test_pred_dict.keys():

   test_stamps = test_stamps_dict[ses_id]
   test_pred = test_pred_dict[ses_id]
   stamp_hrs = [data_handling.get_dayminutes(t)/60 for t in test_stamps]
   
   for t_hr, v in zip(stamp_hrs, test_pred):
      if t_hr < DAY_HOUR_BLOCKS['morning'][1]:
         box_lists[ses_id]['morning'].append(v)
      elif t_hr < DAY_HOUR_BLOCKS['afternoon1'][1]:
         box_lists[ses_id]['afternoon1'].append(v)
      elif t_hr < DAY_HOUR_BLOCKS['afternoon2'][1]:
         box_lists[ses_id]['afternoon2'].append(v)
      else:
         box_lists[ses_id]['evening'].append(v)

boxparams = {
   'ses01': {'widths': .2, 'positions': np.arange(.0, len(DAY_HOUR_BLOCKS)),},
   'ses02': {'widths': .2, 'positions': np.arange(.2, len(DAY_HOUR_BLOCKS)),},
   'ses03': {'widths': .2, 'positions': np.arange(.4, len(DAY_HOUR_BLOCKS)),}
}
FS=14

fig, ax = plt.subplots(1, 1, figsize=(8, 4))

for i_ses, ses_id in enumerate(box_lists):
   ses_lists = [l for l in box_lists[ses_id].values()]
   bp = ax.boxplot(ses_lists, **boxparams[ses_id],
                   patch_artist=True, label=ses_id)
   for patch in bp['boxes']:
      patch.set_facecolor(SES_COLORS[ses_id])
      patch.set_alpha(.4)

ax.legend(ncol=3, frameon=False, fontsize=FS,
          loc='lower center', bbox_to_anchor=(.5, 1))
ax.set_xticks(list(boxparams.values())[1]['positions'])
ax.set_xticklabels(list(DAY_HOUR_BLOCKS.keys()))
ax.set_ylabel('Dyskinesia prediction\n(EMA scale)', fontsize=FS,)
ax.tick_params(labelsize=FS, size=FS,)

plt.tight_layout()

figname = 'lidPred_hm24_allSess_daytimeBlocks'
plt.savefig(os.path.join(ntrl_fig_path, 'proof_kin_pred', figname),
            facecolor='w', dpi=300,)

plt.close()